# The island of knights and knaves

## An agent harness from first principles

In this notebook we build an AI agent that solves a riddle.

The riddle is a classic. On a certain island, every inhabitant is either a **knight**, who always tells the truth, or a **knave**, who always lies. From what the inhabitants say, we can work out who is who.

The riddles come from a standard open source dataset, so the notebook runs anywhere Colab runs, with no website to depend on. Everything else is like the maze notebook: free models through OpenRouter, one tool to see the riddle, one tool to answer it, and a loop that keeps the agent going.

This should be fun regardless of your level of experience.

## Setup part 1: create your OpenRouter account

OpenRouter is a single service that gives you access to hundreds of models, including free ones. We use it throughout the course.

If you already have an OpenRouter account, sign in and go straight to part 2.

1. Open a new browser tab and go to **https://openrouter.ai**
2. Click the **Sign Up** button at the top right of the page.
3. Create your account with the Google button, the GitHub button or your email address. If you choose email, click the link in the verification email that OpenRouter sends you.
4. You do not need to add a credit card or buy credits for this notebook.

## Setup part 2: allow free models in your privacy settings

Free models will not run until you have done this step.

The providers of free models may train on your prompts or publish them. Nothing we send in this course is private.

1. Make sure you are signed in to OpenRouter, then go to **https://openrouter.ai/settings/privacy**
2. Find the setting called **Enable free endpoints that may train on inputs** and switch it **ON**.
3. Find the setting called **Enable free endpoints that may publish prompts** and switch it **ON**.

## Setup part 3: create your API key

If you already have an OpenRouter key saved from another course, use it and go straight to part 4.

1. Go to **https://openrouter.ai/settings/keys**
2. Click the **Create API Key** button.
3. In the **Name** box, type `fde-course`. Leave everything else as it is.
4. Click **Create**.
5. Your new key appears on the screen. It is a long piece of text that starts with `sk-or-v1-`
6. Copy the whole key using the copy button next to it. **OpenRouter only shows you the key once.** Keep this tab open until you have finished part 4. If you lose the key, delete it on this page and create another one.

Never paste your key into a notebook cell. The next step stores it in Colab secrets instead.

## Setup part 4: add the key to Colab as a secret

1. Come back to this notebook in Colab.
2. Look at the narrow strip of icons down the **left edge** of the Colab window. Click the icon that looks like a **key**. A panel called **Secrets** opens.
3. Click **Add new secret**.
4. In the **Name** box, type this exactly, in capital letters, with underscores and with no spaces:

   `OPENROUTER_API_KEY`

5. Click in the **Value** box and paste your key (Ctrl+V on a PC, Cmd+V on a Mac).
6. Switch **ON** the **Notebook access** toggle on the same row.
7. Close the Secrets panel by clicking the key icon again.

## Now for the code

Run each cell in turn by clicking into it and pressing **Shift + Enter**.

In [1]:
# The riddles live on Hugging Face, so we need their datasets library.
# If Colab shows a Restart session button after this cell, click it, then carry on below.
%pip install -q -U datasets


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re

import requests
from datasets import load_dataset
from google.colab import userdata
from IPython.display import HTML, Markdown, display
from openai import OpenAI

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Check your key

This cell reads your key from the Colab secret. If Colab asks whether to grant this notebook access to your secret, click **Grant access**.

If you see **SecretNotFoundError**, check that the secret's name is exactly `OPENROUTER_API_KEY` (Setup part 4).

In [3]:
openrouter_api_key = userdata.get("OPENROUTER_API_KEY").strip()

if openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("This does not look like an OpenRouter API key. Please repeat Setup parts 3 and 4.")

OpenRouter API key exists and begins sk-or-v1-


### Connect to OpenRouter

We use the OpenAI Python package and point it at OpenRouter by changing the base URL.

The model `openrouter/free` picks one of the available free models for each request. We print the name of the model that answered.

In [4]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL, timeout=120, max_retries=5)

# Free accounts get about 50 requests a day to free models. We count every one the agent loop spends.
USAGE = {"requests": 0}

In [5]:
messages = [{"role": "user", "content": "Hi there!"}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by dots-studio/dots-3-note-preview:free




Hi there! I'm dots, an AI model developed by Dots Studio at rednote. How can I help you today?

If you see a reply, everything is set up.

If you see an error that mentions **data policy** or **No endpoints found**, go back to Setup part 2 and check that both switches are on. If you see an error that mentions **401**, repeat Setup parts 3 and 4 to get a new key.

## Part 1 of the harness: the tools

A tool is a Python function that the model can ask us to run.

Our riddles come from **Knights and Knaves**, a standard open source dataset on Hugging Face. Each riddle describes a small island where every inhabitant is either a **knight**, who always tells the truth, or a **knave**, who always lies.

Let us look at one riddle, straight from the dataset.

In [6]:
# 8 inhabitants per riddle: the island where the study's accuracy collapse lives.
# "2ppl" to "3ppl" are the gentler islands, if you want to watch real logic first.
RIDDLES = load_dataset("K-and-K/knights-and-knaves", "test", split="8ppl")

print(f"Loaded {len(RIDDLES)} riddles")
print()
sample = RIDDLES[0]
print(sample["quiz"])
print()
print(f"Inhabitants: {sample['names']}")
print(f"The answer, which the dataset knows and we will keep from the model: {sample['solution']}")

Generating 2ppl split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating 2ppl split: 100%|██████████| 100/100 [00:00<00:00, 1704.18 examples/s]

Generating 3ppl split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating 3ppl split: 100%|██████████| 100/100 [00:00<00:00, 952.95 examples/s]

Generating 3ppl split: 100%|██████████| 100/100 [00:00<00:00, 920.59 examples/s]

...

Generating 7ppl split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating 7ppl split: 100%|██████████| 100/100 [00:00<00:00, 8069.38 examples/s]

Generating 8ppl split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating 8ppl split: 100%|██████████| 100/100 [00:00<00:00, 4972.80 examples/s]

Loaded 100 riddles

A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie. You meet 8 inhabitants: Ethan, Benjamin, Isabella, Zoey, Joseph, Henry, Luke, and Lily. In Ethan's words: "Benjamin is a knight or Zoey is a knight". Benjamin commented, "Ethan is a knight and Henry is a knight". "Benjamin is a knight if and only if Lily is a knave," Isabella claimed. "If Lily is a knave then Luke is a knight," Zoey claimed. According to Joseph, "Lily is a knight if and only if Luke is a knight". "Luke is a knave if and only if Isabella is a knave" - Henry. "Luke is a knight if and only if Zoey is a knave," Luke declared. According to Lily, "Luke is a knight". So who is a knight and who is a knave?

Inhabitants: ['Ethan', 'Benjamin', 'Isabella', 'Zoey', 'Joseph', 'Henry', 'Luke', 'Lily']
The answer, which the dataset knows and we will keep from the model: [False, False, False, False, True, True, False, False]


The dataset knows every answer. Our tools will not tell.

The model never sees this notebook's code, the dataset, or the answers. It only gets a riddle when it asks for one, and its verdict is checked against the dataset's answer with an exact comparison. No guessing about what counts as a correct answer, and no second model acting as a judge.

These are our two tools:

- `get_riddle` hands out the next riddle. The answer stays behind, in our Python variables.
- `submit_answer` takes the agent's verdict, one knight-or-knave call per inhabitant, and names the inhabitants it got wrong.

Small models do not always send tool arguments as tidy JSON, so `submit_answer` accepts the verdict as JSON or as plain text like `Zoey: knave; Oliver: knight`.

### The stage

The tools talk to the model in plain JSON. That is exactly what the model needs, and it is dull to watch.

So before we write the tools, we build the scenery they will draw: a card when a riddle is handed out, a verdict board after every attempt, and a small scoreboard. You will see these appear as the agent works. The model never sees any of it. All it ever gets is JSON.

In [7]:
GAME_CSS = """
<style>
.island-card { border: 2px solid #2a7f62; border-radius: 14px; padding: 12px 18px;
               background: linear-gradient(160deg, #eefaf4, #d9f0e4); max-width: 720px; }
.island-title { font-weight: bold; color: #1d5c48; margin: 0 0 6px 0; }
.island-riddle { font-size: 15px; line-height: 1.5; white-space: pre-wrap; }
.inhabitant { display: inline-block; background: #ffffff; border: 1px solid #bfe3d4;
              border-radius: 999px; padding: 3px 12px; margin: 4px 6px 0 0; }
.good { color: #1d7a43; font-weight: bold; }
.bad { color: #b3261e; font-weight: bold; }
.scoreboard { margin-top: 8px; padding-top: 6px; border-top: 1px dashed #bfe3d4;
              color: #44534d; font-size: 14px; }
.hint { margin-top: 8px; color: #7a5a17; font-size: 14px; }
.board { border-collapse: collapse; margin-top: 10px; }
.board th, .board td { border: 1px solid #bfe3d4; padding: 6px 12px; font-size: 14px; text-align: left; }
.board th { background: #ffffff; color: #1d5c48; }
</style>"""

def esc(text):
    """Make any text safe to place inside HTML"""
    return str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")

def show(html):
    """Draw one frame of the game for the humans watching. The model never sees it"""
    display(HTML(GAME_CSS + html))

def game_header():
    """The title card at the start of a run"""
    show("""
<div class="island-card">
  <p class="island-title">The island of knights and knaves</p>
  <div class="island-riddle">Knights always tell the truth. Knaves always lie.<br>
  Work out who is who, and the island is yours.</div>
</div>""")

def island_card(riddle_id, inhabitants, riddle):
    """The card for a freshly handed-out riddle"""
    who = " ".join(f'<span class="inhabitant">{esc(name)}</span>' for name in inhabitants)
    show(f"""
<div class="island-card">
  <p class="island-title">Riddle {riddle_id} — who tells the truth?</p>
  <div class="island-riddle">{esc(riddle)}</div>
  <div>{who}</div>
</div>""")

def verdict_board(riddle_id, verdict, gold, wrong, attempts, hint=None):
    """The verdict board after every attempt. Failed boards hide the truth; the elder's hint is the only reveal"""
    solved = not wrong
    chips = []
    for name, is_knight in gold.items():
        said = str(verdict.get(name, "")).strip().lower() or "no verdict"
        if solved:
            truth = "knight" if is_knight else "knave"
            chips.append(f'<span class="inhabitant">{esc(name)}: {truth} <span class="good">right</span></span>')
        elif name in wrong:
            chips.append(f'<span class="inhabitant">{esc(name)}: said {esc(said)} <span class="bad">wrong</span></span>')
        else:
            chips.append(f'<span class="inhabitant">{esc(name)}: said {esc(said)} <span class="good">right</span></span>')
    if solved:
        attempts_word = "attempt" if attempts == 1 else "attempts"
        title = f'Riddle {riddle_id} solved in {attempts} {attempts_word}!'
        hint_line = ""
    else:
        title = f'Attempt {attempts}: the island shakes its head'
        if hint:
            hint_line = f'<div class="hint">The island elder whispers: <b>{esc(hint["name"])} is a {esc(hint["is"])}</b>.</div>'
        else:
            hint_line = '<div class="hint">The elder has no more hints. Assemble what it told you.</div>'
    chips_html = "".join(chips)
    show(f"""
<div class="island-card">
  <p class="island-title">{title}</p>
  <div>{chips_html}</div>
  {hint_line}
  {score_line()}
</div>""")

def score_line():
    """One line of scoreboard, reused everywhere"""
    served = len(_solutions)
    return (f'<div class="scoreboard">solved {_score["solved"]}'
            f' &nbsp;·&nbsp; riddles served {served}'
            f' &nbsp;·&nbsp; attempts {_score["attempts"]}'
            f' &nbsp;·&nbsp; hints {_score["hints"]}</div>')

In [8]:
# What the island knows, kept away from the model
_solutions = {}
_attempts = {}
_hints = {}
_solved = set()
_score = {"solved": 0, "attempts": 0, "hints": 0}

def parse_assignment(value):
    """Accept a verdict as JSON or as plain text, and return a dict"""
    if isinstance(value, dict):
        return value
    value = str(value).strip()
    if value.startswith("{") and value.endswith("}"):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            parsed = None
        if isinstance(parsed, dict):
            return parsed
    assignment = {}
    for part in re.split(r"[;,\n]", str(value)):
        if ":" in part:
            name, role = part.split(":", 1)
            assignment[name.strip(" .-'")] = role.strip(" .-'").lower()
    return assignment

def get_riddle():
    """Hand out the next riddle. The answer stays in our records"""
    number = len(_solutions) + 1
    if number > len(RIDDLES):
        return {"message": "The island has no riddles left."}
    sample = RIDDLES[number - 1]
    _solutions[number] = dict(zip(sample["names"], sample["solution"]))
    island_card(number, sample["names"], sample["quiz"])
    return {"riddle_id": number, "inhabitants": sample["names"], "riddle": sample["quiz"]}

def submit_answer(riddle_id, assignment):
    """Check a verdict against the island's records and name the inhabitants we got wrong"""
    rid = int(str(riddle_id).strip())
    verdict = parse_assignment(assignment)
    gold = _solutions[rid]
    lookup = {str(name).strip(" .-'").lower(): str(role) for name, role in verdict.items()}
    said_for = {name: str(lookup.get(name.lower(), "")).strip(" .-'").lower() for name in gold}
    wrong = []
    for name, is_knight in gold.items():
        truth = "knight" if is_knight else "knave"
        if said_for[name] != truth:
            wrong.append(name)
    _attempts[rid] = _attempts.get(rid, 0) + 1
    _score["attempts"] += 1
    hint = None
    if wrong:
        revealed = _hints.setdefault(rid, [])
        pending = [name for name in wrong if name not in revealed]
        if pending:
            elder_choice = pending[0]
            revealed.append(elder_choice)
            _score["hints"] += 1
            hint = {"name": elder_choice, "is": "knight" if gold[elder_choice] else "knave"}
        result = {
            "solved": False,
            "attempt": _attempts[rid],
            "judged_wrong": wrong,
            "message": "The island shakes its head. Think about these inhabitants again and submit a new verdict.",
        }
        if hint:
            result["hint"] = f'The island elder whispers: {hint["name"]} is a {hint["is"]}.'
        else:
            result["message"] = "The elder has no more hints. It has judged you wrong on these names before; use what it told you."
    else:
        if rid not in _solved:
            _solved.add(rid)
            _score["solved"] += 1
        result = {
            "solved": True,
            "attempt": _attempts[rid],
            "message": "Every inhabitant is judged correctly. The island's riddle is solved!",
        }
    verdict_board(rid, said_for, gold, wrong, _attempts[rid], hint)
    return result

## Trying the tools ourselves

Before we let the model anywhere near them, we run the tools the way the model would.

In [9]:
print(get_riddle())

{'riddle_id': 1, 'inhabitants': ['Ethan', 'Benjamin', 'Isabella', 'Zoey', 'Joseph', 'Henry', 'Luke', 'Lily'], 'riddle': 'A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie. You meet 8 inhabitants: Ethan, Benjamin, Isabella, Zoey, Joseph, Henry, Luke, and Lily. In Ethan\'s words: "Benjamin is a knight or Zoey is a knight". Benjamin commented, "Ethan is a knight and Henry is a knight". "Benjamin is a knight if and only if Lily is a knave," Isabella claimed. "If Lily is a knave then Luke is a knight," Zoey claimed. According to Joseph, "Lily is a knight if and only if Luke is a knight". "Luke is a knave if and only if Isabella is a knave" - Henry. "Luke is a knight if and only if Zoey is a knave," Luke declared. According to Lily, "Luke is a knight". So who is a knight and who is a knave?'}


In [10]:
# A deliberately lazy verdict: we claim that everyone is a knave
everyone_is_a_knave = {name: "knave" for name in _solutions[1]}
print(submit_answer(1, everyone_is_a_knave))

{'solved': False, 'attempt': 1, 'judged_wrong': ['Joseph', 'Henry'], 'message': 'The island shakes its head. Think about these inhabitants again and submit a new verdict.', 'hint': 'The island elder whispers: Joseph is a knight.'}


The model never sees our Python code. We describe each tool to it in JSON with the tool's name, what it does and the arguments it takes. The model reads these descriptions and decides when to use the tools.

For `submit_answer` we spell out the only two words allowed per inhabitant. Small models stick to the list far better than to a free-form sentence.

In [11]:
get_riddle_json = {
    "type": "function",
    "function": {
        "name": "get_riddle",
        "description": "Receive the next riddle from the island of knights and knaves",
        "parameters": {"type": "object", "properties": {}},
    },
}

submit_answer_json = {
    "type": "function",
    "function": {
        "name": "submit_answer",
        "description": "Submit your verdict for a riddle: for every inhabitant, whether they are a knight or a knave",
        "parameters": {
            "type": "object",
            "properties": {
                "riddle_id": {"type": "integer", "description": "The id of the riddle, from get_riddle"},
                "assignment": {
                    "type": "object",
                    "description": "One entry per inhabitant: their name and whether they are a knight or a knave",
                    "additionalProperties": {"type": "string", "enum": ["knight", "knave"]},
                },
            },
            "required": ["riddle_id", "assignment"],
        },
    },
}

tools = [get_riddle_json, submit_answer_json]

When the model wants to use a tool, it replies with the tool's name and the arguments as JSON. Our code runs the function and sends back the result.

If the tool fails, we send the error message back to the model as the result. The result of each tool is a dict, so we send it as JSON text.

In [12]:
TOOLS = {"get_riddle": get_riddle, "submit_answer": submit_answer}

def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = TOOLS[tool_call.function.name](**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    content = result if isinstance(result, str) else json.dumps(result)
    return {"role": "tool", "tool_call_id": tool_call.id, "content": content}

## Using the tools once

Now we call the model and ask it to fetch the island's riddle. The model replies with a request to use the tool.

In [13]:
messages = [{"role": "user", "content": "Please call get_riddle to receive the island's riddle. Do not try to solve it yet. Just tell me who the inhabitants are."}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

Answered by nvidia/nemotron-3.5-super-vl:free
Finish reason: tool_calls
[ChatCompletionMessageFunctionToolCall(id='call-3825c900-092f-4790-9e5b-0f98c1315227', function=Function(arguments='{}', name='get_riddle'), type='function', index=0)]


The finish reason is `tool_calls`. The model has asked for `get_riddle`.

We add the model's request and the tool's result to the conversation and call the model again.

Now and then a small model asks for the tool again instead of answering. If the output is blank, run the two cells again.

In [14]:
messages.append(message)
if message.tool_calls:
    for tool_call in message.tool_calls:
        messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by cohere/north-mini-code:free


The inhabitants of the island are:

- Luke
- Isabella  
- Jackson
- Aria
- Mason
- Harper
- Abigail
- Emily

## Part 2 of the harness: the loop

First, a system prompt that asks the model to reply in Markdown.

In [15]:
SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Your replies are displayed as Markdown, so format them in Markdown with headings, bold text and lists.
Put any map or diagram in a code block so that it lines up. Do not use emojis.
"""

TASK = "Call get_riddle to hear the island's riddle. Work out who is a knight and who is a knave, then call submit_answer with your verdict. Keep going until the island accepts your answer."

The agent is the code from the last section inside a loop. The loop stops when the model replies with text and no tool call, or after 25 steps. If a free model sends back an empty reply, we ask again.

In [16]:
def run_agent(task, max_steps=25, model=MODEL):
    """Call the model in a loop, running the tools whenever it asks, until it gives a final answer"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        USAGE["requests"] += 1
        response = openrouter.chat.completions.create(model=model, messages=messages, tools=tools)
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return step
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

## Onto the island

Everything is ready. The island hands out one riddle at a time.

We give the agent one instruction. A run takes a minute or two. If the agent submits a wrong verdict, that is part of the show: the island names the inhabitants it judged wrong, and its elder whispers one true role — a fresh hint on every failed attempt. If you see an error, or the agent gives up, run the cell again.

In [17]:
game_header()
run_agent(TASK)

Step 1 answered by nvidia/nemotron-3.5-lightning:free


Step 2 answered by nvidia/nemotron-3.5-super-vl:free


Step 3 answered by poolside/laguna-s-2.1:free


The island accepted my answer on the first attempt. Here's how I solved it:

## Reasoning

Let `K(x)` mean "x is a knight". Writing each statement as a logical constraint:

- **Isabella** said Matthew is a knight: `K(I) ↔ K(M)`
- **Avery** said "If Ella is a knave then Matthew is a knave": `K(A) ↔ (K(E) ∨ ¬K(M))`
- **Ella** said Luke is a knight: `K(E) ↔ K(L)`
- **Matthew** said "Avery is a knight or Matthew is a knight": reduces to `K(A) → K(M)`
- **Luke** said "If Avery is a knave then Luke is a knight": reduces to `K(A) → K(L)`
- **Mia** said "If Ella is a knave then Benjamin is a knave": `K(Mi) ↔ (K(E) ∨ ¬K(B))`
- **Scarlett** said Isabella is not a knave: `K(S) ↔ K(I)`
- **Benjamin** said Mia is not a knave: `K(B) ↔ K(Mi)`

Key chain: Mia's statement `K(Mi) ↔ (K(E) ∨ ¬K(Mi))` forces **Mia = knight** (assuming she's a knave makes the equation false). Then `K(E) ∨ ¬K(Mi)` being true means `K(E)` is true, so Luke is a knight. From there the constraints cascade:

- `K(Mi) → K(B)` → Benjamin is a knight
- `K(E)` true → Avery's statement is true → Avery is a knight
- `K(A)` true → both Matthew and Isabella must be knights
- `K(I)` true → Scarlett is a knight

**Verdict:** Every inhabitant is a knight — and all statements are consistent under this assignment.

3

## What just happened

The model asked for a riddle, read what the inhabitants said, made a first verdict, heard which inhabitants it had wrong, reasoned again and submitted until the island accepted.

Several different models probably took turns answering. Each one worked from the same conversation history. The dataset and its answers stayed on our side of the glass the whole time: the only way in was through the tools. The cards and the scoreboard were drawn by our code, for us. The model never saw them; all it ever got was JSON.

This notebook loads the eight-inhabitant island, where the study's accuracy collapse lives. Almost nothing solves it alone: the verdicts keep coming back wrong, and the win gets assembled out of the elder's hints instead. Watch for the moment the reasoning stops and the hint mining starts. That is not cheating — that is the harness doing its job, turning a puzzle the model cannot solve into a game it can. Switch the dataset cell to `"2ppl"` or `"3ppl"` to watch the same models do the logic by themselves.

The code around the model is called the agent harness. It is the same harness the maze agent used, one tool to look and one loop to keep going.

### Try it yourself

- Run the last cell again. The island serves the next riddle. Does the agent need fewer attempts?
- The island in the dataset cell is `"8ppl"`, the steepest cliff in the study. Drop it to `"3ppl"` and rerun to watch the same models solve with logic instead of hints.
- The tournament discovers the whole chat-capable free slate at runtime and sizes today's heat to your budget — no hardcoded list. To pin the field by hand, overwrite `all_models` in the setup cell.
- Give the island a third tool, `get_hint(riddle_id)`, that reveals one inhabitant's true role. The model can then spend a tool call to buy a clue. You write the tool, the model decides when to use it.

## Part 3: the tournament

One riddle, every free model on OpenRouter, the same tools. Who really solves the game?

The router `openrouter/free` picks whichever free model is awake, so no two runs are comparable. Nothing is hardcoded either: the setup cell asks OpenRouter what is free right now, keeps the models a human can chat with (the router itself, the safety classifier and the music models are out), and ranks them by OpenRouter's own aggregated intelligence index, then by context window, then by recency.

The heat is sized to the day automatically, by two rules: the tournament never outgrows the square of the day's budget (N x N stays under it), and every contestant keeps room for a worst-case run of up to 14 steps. On a free account — about **50 requests a day**, or about **1000 once you have added $10 of credits** — that is three models per day; on the credits tier the square rule allows up to 31, so the whole free slate runs in one sitting. Everyone left over waits in the printed queue for tomorrow.

One wrinkle needs one ping: a few free models only answer through verified agentic harnesses, and OpenRouter politely refuses everyone else with a 403. So before anyone runs, each contestant gets a one-token ping. A gated model is recorded as an error, its heat slot goes to the next model in the queue, and the tournament stays the size the budget promised.

The setup cell shows the discovery, the sizing and the spend before it calls any model, and the leaderboard shows the running total.

In [18]:
# Nothing hardcoded: ask OpenRouter what is free right now and rank it.
# (To pin the field by hand for a lesson: all_models = ["nvidia/nemotron-3-ultra-550b-a55b:free", ...])
import math

DAILY_BUDGET = 50    # free accounts: about 50 free-model requests a day (about 1000 once you have added $10 of credits)
STEPS_ALLOWED = 14   # an 8-inhabitant riddle can need 8 elder hints plus a final verdict

catalog = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]

def discover_contestants(catalog):
    """Every chat-capable free model on OpenRouter right now, strongest-looking first"""
    roster = []
    for m in catalog:
        price = m.get("pricing", {})
        if float(price.get("prompt", 1)) != 0.0 or float(price.get("completion", 1)) != 0.0:
            continue  # not free
        if m["id"] == "openrouter/free":
            continue  # the router itself
        outputs = set(m.get("architecture", {}).get("modality", "").split("->")[-1].split("+"))
        if outputs != {"text"}:
            continue  # music, image and other outputs nobody can chat with
        blob = f'{m["id"]} {m.get("name", "")}'.lower()
        if "safety" in blob or "moderation" in blob:
            continue  # classifiers, not contestants
        roster.append(m)
    def strength(m):
        b = (m.get("benchmarks") or {}).get("artificial_analysis") or {}
        return (-(b.get("intelligence_index") or -1), -(m.get("context_length") or 0), -m.get("created", 0))
    roster.sort(key=strength)
    return [m["id"] for m in roster]

all_models = discover_contestants(catalog)
if len(all_models) < 2:
    raise SystemExit("Almost nothing is free on OpenRouter right now. Run this cell again later.")

# Size today's heat: never more models than the square root of the day's budget (N x N stays under it),
# and room for one worst-case run per contestant (+1 for a one-token ping that catches harness-gated models).
square_cap = math.isqrt(DAILY_BUDGET - 1)
fits_today = max(0, (DAILY_BUDGET - USAGE["requests"]) // (STEPS_ALLOWED + 1))
heat = all_models[:min(len(all_models), square_cap, fits_today)]
queue = [m for m in all_models if m not in heat]

print(f"Discovered {len(all_models)} chat-capable free models. Today's heat: {len(heat)}, square cap: {square_cap}")
for name in heat:
    print(f"  today:  {name}")
for name in queue:
    print(f"  queued: {name}")
print(f"The agent loop has spent {USAGE['requests']} requests of about {DAILY_BUDGET} a day.")

Discovered 15 chat-capable free models. Today's heat: 3, square cap: 7
  today:  nvidia/nemotron-3.5-super-vl:free
  today:  thinkingmachines/inkling-small:free
  today:  thinkingmachines/inkling:free
  queued: nvidia/nemotron-3-ultra-550b-a55b:free
  queued: nvidia/nemotron-3.5-lightning:free
  queued: dots-studio/dots-3-note-preview:free
  queued: recursive/flint-v1.0:free
  queued: xingchen-agi/xing4.0-29b-a4b:free
  queued: poolside/laguna-s-2.1:free
  queued: poolside/laguna-xs-2.1:free
  queued: google/gemma-4-26b-a4b-it:free
  queued: google/gemma-4-31b-it:free
  queued: cohere/north-mini-code:free
  queued: nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
  queued: liquid/lfm-2.5-2.6b:free
The agent loop has spent 3 requests of about 50 a day.


In [19]:
import time

def reset_island():
    """Wipe the island's records so every contestant meets the same fresh riddle"""
    _solutions.clear()
    _attempts.clear()
    _hints.clear()
    _solved.clear()
    _score.update(solved=0, attempts=0, hints=0)

def leaderboard(results):
    """The final table: who really solved the game"""
    ranked = sorted(results, key=lambda r: (r.get("error") is not None, not r["solved"], r["attempts"], r["steps"]))
    ranks = ["1st", "2nd", "3rd"]
    rows = []
    for i, r in enumerate(ranked):
        if r.get("error"):
            rank = "-"
            verdict = f'<span class="bad" title="{esc(r["error"])}">error</span>'
        elif r["solved"]:
            rank = ranks[i] if i < 3 else ""
            verdict = "solved"
        else:
            rank = "-"
            verdict = "did not"
        rows.append(f"<tr><td>{rank}</td><td>{esc(r['model'])}</td><td>{verdict}</td>"
                    f"<td>{r['attempts']}</td><td>{r['hints']}</td><td>{r['steps']}</td></tr>")
    rows_html = "".join(rows)
    show(f"""
<div class="island-card">
  <p class="island-title">Who really solved the game</p>
  <table class="board">
    <tr><th>Rank</th><th>Model</th><th>Verdict</th><th>Attempts</th><th>Hints</th><th>Steps</th></tr>
    {rows_html}
  </table>
  <div class="scoreboard">requests spent by the agent loop: {USAGE["requests"]} of about {DAILY_BUDGET} a day</div>
</div>""")

# Some free models only answer through verified agentic harnesses; OpenRouter politely 403s everyone else.
# One tiny ping per model catches the gate before it eats a heat slot, and the next in line steps up.
def answers_directly(name):
    """One tiny request: does this model talk to a plain notebook, or is it harness-gated?"""
    try:
        openrouter.chat.completions.create(model=name, messages=[{"role": "user", "content": "ping"}], max_tokens=1)
        return True
    except Exception as error:
        if getattr(error, "status_code", None) == 403:
            return False  # the agentic-harness gate: permanent for a notebook
        return True  # busy pools and hiccups are worth trying inside the run itself

results = []
roster = list(heat) + list(queue)
final_heat = []
while roster and len(final_heat) < len(heat) and USAGE["requests"] + STEPS_ALLOWED <= DAILY_BUDGET:
    name = roster.pop(0)
    USAGE["requests"] += 1
    if answers_directly(name):
        final_heat.append(name)
    else:
        USAGE["requests"] -= 1  # a refused ping got no words back, so it does not cost a request
        print(f"{name} is harness-gated (403) — recorded as an error, the next model steps up")
        results.append({"model": name, "solved": False, "attempts": 0, "hints": 0, "steps": 0,
                        "error": "403 — only available through agentic harnesses"})

for name in final_heat:
    # A refused ping never got a word back from any model, so it does not eat a contestant's slot.
    if USAGE["requests"] + STEPS_ALLOWED > DAILY_BUDGET:
        print(f"Stopping before {name}: the next run could pass about {DAILY_BUDGET} requests for today.")
        print("Tomorrow: run the setup cell again — it discovers the field and the queue steps up.")
        break
    reset_island()
    print(f"--- {name} steps up to the island ---")
    try:
        steps = run_agent(TASK, max_steps=STEPS_ALLOWED, model=name)
        results.append({
            "model": name,
            "solved": _score["solved"] > 0,
            "attempts": _score["attempts"],
            "hints": _score["hints"],
            "steps": steps if steps else STEPS_ALLOWED,
            "error": None,
        })
    except Exception as error:
        print(f"{name} could not finish: {error}")
        results.append({
            "model": name,
            "solved": False,
            "attempts": _score["attempts"],
            "hints": _score["hints"],
            "steps": STEPS_ALLOWED,
            "error": str(error)[:100],
        })
    time.sleep(3)  # free models share a small per-minute budget; give the next one room

leaderboard(results)

# Upstream pools (Google AI Studio and friends) throttle now and then, independent of OpenRouter's daily count.
# To give the unlucky ones another shot, run the setup and run cells again tomorrow — the sizing finds room.

thinkingmachines/inkling-small:free is harness-gated (403) — recorded as an error, the next model steps up
thinkingmachines/inkling:free is harness-gated (403) — recorded as an error, the next model steps up


--- nvidia/nemotron-3.5-super-vl:free steps up to the island ---


Step 1 answered by nvidia/nemotron-3.5-super-vl:free


Step 2 answered by nvidia/nemotron-3.5-super-vl:free


I'll work through the logic:

**Key deductions:**
1. **Lily** says "Luke is a knight" → Lily is knight iff Luke is knight (L ↔ Lu).
2. **Joseph** says "Lily is knight iff Luke is knight" — since L ↔ Lu always holds, Joseph's statement is always true → **Joseph is a knight**.
3. **Luke** says "Luke is knight iff Zoey is knave" (Lu ↔ ¬Z). Testing both values of Lu forces **Zoey is a knave** in all cases.
4. Zoey (knave) says "If Lily is knave then Luke is knight" (¬L → Lu) must be false → **Lily is knave** and **Luke is knave**.
5. With Lily = knave, Luke = knave:
   - **Henry**: "Luke knave iff Isabella knave" → H = ¬I
   - **Isabella**: "Benjamin knight iff Lily knave" → I = B
   - **Ethan**: "Benjamin knight or Zoey knight" (= B) → E = B
   - **Benjamin**: "Ethan knight and Henry knight" → B = (E ∧ H) = (B ∧ ¬B) = false → **Benjamin is knave**

This cascades: E=knave, I=knave, H=knight.

**Verification (all 8 statements check out).**

Final assignment:
- Ethan: knave, Benjamin: knave, Isabella: knave, Zoey: knave, Joseph: knight, Henry: knight, Luke: knave, Lily: knave

...

Step 1 answered by nvidia/nemotron-3.5-lightning:free


Step 2 answered by nvidia/nemotron-3.5-lightning:free


Step 3 answered by nvidia/nemotron-3.5-lightning:free


**Solution**

After working through the statements, the consistent assignments are:

| Inhabitant | Type   |
|------------|--------|
| Ethan      | knave |
| Benjamin   | knave |
| Isabella   | knave |
| Zoey       | knave |
| Joseph     | knight|
| Henry      | knight|
| Luke       | knave |
| Lily       | knave |

All eight statements are satisfied with these assignments, so the island accepts the answer.

Rank,Model,Verdict,Attempts,Hints,Steps
1st,nvidia/nemotron-3.5-super-vl:free,solved,1,0,3
2nd,nvidia/nemotron-3-ultra-550b-a55b:free,solved,1,0,3
3rd,nvidia/nemotron-3.5-lightning:free,solved,1,0,3
-,thinkingmachines/inkling-small:free,error,0,0,0
-,thinkingmachines/inkling:free,error,0,0,0


### Didn't work?

- An error that mentions **data policy** or **No endpoints found** means Setup part 2. An error that mentions **401** means repeat Setup parts 3 and 4. **SecretNotFoundError** means Setup part 4.
- **ModuleNotFoundError: No module named 'datasets'** means the install cell at the top has not run. Run it, then run the cells from the top.
- If the agent submits a verdict the island cannot read, run the cell again. Small models send odd JSON now and then, and the checker accepts plain text as a backup.
- A free OpenRouter account allows about 50 requests a day to free models (about 1000 once you have added $10 of credits). The heat is sized to fit before anything runs — never more models than the square of the budget allows, never more runs than the day can pay for. On a free day that is three models; tomorrow the setup cell discovers the field again and the queue steps up.
- An error that mentions **429** or **rate limit** means the day's free requests are spent. The counters in the tournament cells tell you where you stand. Quotas reset within a day.
- A contestant can retire overnight. The tournament cell skips it and says so; if two or more are gone, pick fresh names from the free models page.
- If the leaderboard shows **error** for a contestant, the reason rides along in the table (hover over it). **429** or **rate limit** means the model's free pool is busy or your day's requests are spent. A **TypeError** about an unexpected argument means an old cell is still loaded in the session — run the notebook from the top.

### Credits

- The riddles come from the open source dataset [K-and-K/knights-and-knaves](https://huggingface.co/datasets/K-and-K/knights-and-knaves) (Xie et al., 2024), shared under CC BY-NC-SA 4.0. Use it for teaching and fun, not for a commercial product. The same study documents the difficulty cliff: accuracy collapses from four inhabitants up.
- Inspired by [the maze notebook](https://colab.research.google.com/drive/1tyDDXRnhdZll3dLpwuo_r3viQL4FzUwo) — "Instant gratification: an agent harness from first principles" — in which an agent escapes a maze of web pages at [maze.edwarddonner.com](https://maze.edwarddonner.com). We took that harness as our starting point and expanded it into a more complicated riddle for the agentic harness: the island, the game stage and the tournament of free models all grew on top of it.